# Proxy V4 — tiếp tục VideoSwin từ epoch 5 đến epoch 10

Add Input: Kinetics cleaned; Output V7 có recovery_split.json; Output Proxy V4 có large_reaudit/best_feasible_reaudit.pt; Output run 5 epoch có v4_swin_rate_recovery_*/swin_ratio_095/last.pt; và checking có v5_fixed_split/split_manifest.json. Bật GPU và Internet rồi chọn Run All.

In [ ]:
%cd /kaggle/working
from pathlib import Path
import subprocess
import sys

PROJECT = Path('/kaggle/working/proxy_v4')
if not PROJECT.is_dir():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/munnn01/proxy_v4.git', str(PROJECT)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '-q', '--ff-only', 'origin', 'main'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '-r', str(PROJECT / 'requirements.txt')], check=True)
%cd /kaggle/working/proxy_v4


In [ ]:
from kaggle_cells.v4_swin_continue import prepare

STAGE, RESUME_CHECKPOINT = prepare()
print('Resume checkpoint:', RESUME_CHECKPOINT)


In [ ]:
from kaggle_cells.v4_swin_continue import continue_training

CANDIDATE, FEASIBLE = continue_training(STAGE, RESUME_CHECKPOINT)
print('Candidate:', CANDIDATE)
print('Feasible:', FEASIBLE)


In [ ]:
from kaggle_cells.v4_swin_rate_recovery import evaluate_full

EVAL_DIR = evaluate_full(STAGE, CANDIDATE) if FEASIBLE else None
if EVAL_DIR is None:
    print('Chưa feasible; không chạy full evaluation.')
else:
    print('Full evaluation:', EVAL_DIR)


In [ ]:
from IPython.display import FileLink, display

display(FileLink(str(STAGE.root / 'continuation_selection.json')))
display(FileLink(str(STAGE.root / 'continuation_result.json')))
display(FileLink(str(CANDIDATE)))
if EVAL_DIR is not None:
    display(FileLink(str(EVAL_DIR / 'metrics.csv')))
    display(FileLink(str(EVAL_DIR / 'bd_rate.json')))
    display(FileLink(str(EVAL_DIR / 'goal_check.json')))
    display(FileLink(str(EVAL_DIR / 'h264_top1_bpp_bd_rate.png')))
